>Introduction

Climate impact studies in mountain regions need information at scales of a few kilometres, where elevation, slope and aspect control temperature and precipitation. Global climate models (GCMs) solve the equations of atmospheric, oceanic and land-surface dynamics to simulate the climate system and project its response to greenhouse gas forcing [1], To make global simulations computationally feasible, they use grids of 100-200km. The model HadGEM2-ES, for example, has a resolution of 1.875º x 1.25º, about 140 km over northern Spain. At that resolution, the Pyrenees appear as a smooth ridge.

>Downscaling

Downscaling bridges the gap between global and local scales. Dynamical downscaling nests a regional climate model inside the GCM. It resolves small-scale physics explicitly, but it inherits the biases of the driving model and is computationally expensive. By contrast, statistical downscaling is computationally efficient. It involves deriving empirical relationships linking large-scale atmospheric variables (predictors) and local/regional climate variables (predictands). However, it assumes that the relationships calibrated on the historical climate remain valid in the future.

>Methodology

This project applies the **Multivariate Adaptive Constructed Analogs (MACA)** method (Abatzoglou & Brown, 2012) to downscale daily maximum temperature, minimum temperature and precipitation over the Pyrenees, from the GCM grid (~140 km) to the 0.1° (~10 km) grid of the E-OBS observational dataset (Cornes et al., 2018). MACA combines quantile-mapping bias correction with a constructed-analogs step. The analogs are fine-scale observed patterns, so the downscaled fields inherit realistic spatial structure from real observations.

> 1st stage scope

We use a single GCM, HadGEM2-ES, from its CMIP5 historical simulation. The goal is to develop and validate the method against observations. The model is calibrated on 1950-1980 and evaluated on the independent period 1981-2005. Extending the workflow to future scenarios will require a multi-model ensemble.

> Data extraction

We first downlad the data of the GCMs on the **historical** experiment to perform the statistical downscaling by comparing it with the observed data. We will select the data from this region (see Fig. 1):

<p align="center">
    <img src="../reports/figures/map_ROI.png" width="400">
    <br>
    <i>Fig. 1: Downscale region of interest around the Pyrenees. Square coordinates: (44.5,-2.2) (40.8,3.6)</i>
</p>



In [1]:
from pathlib import Path
RAW = Path('../data/raw')
INTERIM = Path('../data/interim')

In [2]:
import zipfile
out = INTERIM / "gcm_nc"
out.mkdir(parents=True, exist_ok=True)
for f in sorted((RAW / "gcm").glob("*.zip")):
    with zipfile.ZipFile(f) as z:
        z.extractall(out)

In [4]:
import xarray as xr
data = {}
for v in ["pr", "tasmax", "tasmin"]:
    files = sorted(out.glob(f"{v}_*.nc"))
    ds_v = xr.open_mfdataset(files, combine="by_coords",
                             data_vars="minimal", coords="minimal", compat="override")
    data[v] = ds_v[v]

xrds = xr.Dataset(data)
xrds


<xarray.Dataset> Size: 13MB
Dimensions:  (time: 34560, lat: 5, lon: 6)
Coordinates:
  * time     (time) object 276kB 1909-12-01 12:00:00 ... 2005-11-30 12:00:00
  * lat      (lat) float64 40B 40.0 41.25 42.5 43.75 45.0
  * lon      (lon) float64 48B 356.2 358.1 0.0 1.875 3.75 5.625
    height   float64 8B 1.5
Data variables:
    pr       (time, lat, lon) float32 4MB dask.array<chunksize=(1, 5, 6), meta=np.ndarray>
    tasmax   (time, lat, lon) float32 4MB dask.array<chunksize=(1, 5, 6), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 4MB dask.array<chunksize=(1, 5, 6), meta=np.ndarray>

In [5]:
print(xrds.sizes)
print(xrds.time.values[[0, -1]])
print(xrds.lat.values, xrds.lon.values)

Frozen({'time': 34560, 'lat': 5, 'lon': 6})
[cftime.Datetime360Day(1909, 12, 1, 12, 0, 0, 0, has_year_zero=True)
 cftime.Datetime360Day(2005, 11, 30, 12, 0, 0, 0, has_year_zero=True)]
[40.   41.25 42.5  43.75 45.  ] [356.25  358.125   0.      1.875   3.75    5.625]
